# 05 · Validation Plan — dose-response, LOD, controls, multiplexing

**Standard slot:** *validation plan.* **For Project 12 this means:** turn the top integrated
constructs into a **costed, controlled functional-readout plan** — a **luminescence/FRET
dose-response**, an **LOD estimate**, the mandatory controls (**no-analyte / blank** and
**off-target**), an expression strategy, and a **multiplexing** concept `[stretch]` (D4/D5).

A construct that looks good in silico is a **hypothesis** — the dose-response is what tests it. **No
fabricated LOD or luminescence numbers**: you estimate detectability from a *planned* dose-response
with replicates and a blank. Needs `results/top_constructs.csv` (notebook 04).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the functional-readout validation plan

Generate a plan card from the top constructs: the dose-response assay, controls, expression, timeline,
costed reagents. Fill the `<...>` from your own numbers; this is the deliverable other people read.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_constructs.csv") if os.path.exists("results/top_constructs.csv") else pd.DataFrame()
n_top = len(top)
by_fam = top.groupby("family").size().to_dict() if n_top else {}

plan = f"""# De Novo Binder->Biosensor Validation Plan (Project 12 — by <your name>, <date>)

## Analyte + readout
Analyte: <your chosen biomarker> (verify RCSB accession). Readout: <split-luciferase/NanoBiT
luminescence, or split-FP FRET>. Switch family/families carried: {by_fam}.

## Candidates
Top {n_top} integrated constructs carried forward; see results/top_constructs.csv. EVERY in-silico
number is a HYPOTHESIS: pae_interaction is binder confidence (not affinity); dynamic_range is a
MODELED proxy (not measured signal); there is NO measured LOD yet.

## Expression strategy
- Construct (binder + switch fusion): E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC.
  For split-luciferase/NanoBiT, confirm the reporter folds and is active in the fusion context.
- Analyte/biomarker reagent: recombinant (mammalian/insect or commercial); confirm it is the right
  isoform/PTM state for your sensor.

## Functional assay (the point of the project): dose-response
1. Titrate the analyte across a wide concentration range (e.g. log-spaced, >= 8 points + blank).
2. Read the signal: luminescence (split-luciferase/NanoBiT) or FRET ratio (split-FP).
3. Fit signal vs [analyte] (e.g. 4-parameter logistic) -> EC50 + dynamic range (max/min, fold).
4. ESTIMATE the LOD from the fit: blank mean + 3*SD (or the lowest distinguishable dose). This is a
   MEASURED estimate from YOUR data — never a number copied from a model.

## Controls (MANDATORY)
- NO-ANALYTE / BLANK: buffer only -> defines the OFF/background signal and the LOD floor. The single
  most important control for a sensor (a leaky OFF state kills dynamic range).
- OFF-TARGET: a structurally-related but wrong analyte (or an unrelated protein) -> the sensor must
  NOT light up. This is the specificity control.
- POSITIVE: a known concentration of the true analyte (and, if available, an established sensor/ELISA)
  to confirm the assay works and to cross-calibrate.

## Realistic expectations
Coupling binding to a CLEAN ON/OFF signal is hard; dynamic range vs binder affinity is a real
trade-off; MOST integrated constructs need iteration (linker length/rigidity, latch redesign,
reporter placement). Report the dynamic range and LOD you MEASURE, honestly — including constructs
that don't switch. Do NOT imply a working sensor or fabricate an LOD.

## Multiplexing concept [stretch]
Sketch how to detect several analytes at once: orthogonal reporters (different luciferase colors /
FRET pairs), spatial separation (bead/array), or barcoded constructs. Note the cross-talk controls a
multiplex panel needs.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} constructs + off-target/blank controls): $<...>, <...> weeks (IGSC-screened provider).
- Analyte reagent(s) + assay plates + luminometer/plate-reader time: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Diagnostic / point-of-care sensing of a disease biomarker (in scope; low dual-use). Gene synthesis via
a biosecurity-screening provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:600], "...")

## 2 · A simulated dose-response (EXAMPLE_DATA — to design the assay, not to report)

To *plan* the assay you can sketch the expected curve shape from a construct's modeled dynamic range.
This is **EXAMPLE_DATA / SYNTHETIC** — a teaching stand-in for choosing concentrations and replicate
counts, **not** a result. A real curve comes from the lab. We use a simple saturable (Hill) shape.

In [ ]:
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import biosensor_tools as bt

top = pd.read_csv("results/top_constructs.csv") if os.path.exists("results/top_constructs.csv") else pd.DataFrame()
if len(top):
    c = top.iloc[0]
    dr = float(c.get("dynamic_range") or 2.0)
    off = float(c.get("off_signal") or 10.0)
    # EXAMPLE_DATA saturable curve: signal = off * (1 + (dr-1) * x/(x+EC50)); arbitrary EC50.
    EC50 = 5.0   # arbitrary units — the assay measures the real one
    x = np.logspace(-2, 3, 12)
    signal = off * (1.0 + (dr - 1.0) * x / (x + EC50))
    # add a tiny deterministic "assay noise" band for planning replicate counts (SYNTHETIC)
    cv = 0.10
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(x, signal, yerr=cv * signal, fmt="o-", capsize=3)
    ax.set_xscale("log"); ax.set_xlabel("[analyte] (arbitrary units)")
    ax.set_ylabel("signal (a.u., SYNTHETIC)")
    ax.set_title(f"PLANNING dose-response for {c['construct_id'][:24]}... (EXAMPLE_DATA)")
    plt.tight_layout(); plt.savefig("results/p12_doseresponse.png", dpi=150); plt.show()
    lod = bt.estimate_lod(dr, assay_cv=cv)
    print("saved results/p12_doseresponse.png")
    print("LOD PLANNING flag (NOT a measured LOD):", lod)
else:
    print("Run notebook 04 first to produce results/top_constructs.csv.")
print("This curve is EXAMPLE_DATA to size the assay (concentrations, replicates) — never a result.")

## 3 · Off-target specificity controls

The specificity control: the sensor must light up for the **true** analyte and **not** for a related
or unrelated protein. Here we scaffold the panel deterministically; on Colab, run the integrated
construct against each off-target analyte (two-state / AF2) and confirm the dynamic range collapses.

In [ ]:
import biosensor_tools as bt

# Define an off-target panel (EXAMPLE — choose real related/unrelated proteins for your analyte).
OFF_TARGETS = ["RELATED_BIOMARKER_X", "UNRELATED_PROTEIN_Y"]
rows = []
if len(top):
    for _, c in top.head(5).iterrows():
        for ot in OFF_TARGETS:
            # On Colab: re-model the construct with the OFF-TARGET present; dynamic range SHOULD ~1.
            # Mock proxy: an off-target should not switch -> force a near-1 dynamic range (SYNTHETIC).
            rows.append(dict(construct_id=c["construct_id"], off_target=ot,
                             expected_dynamic_range=1.0,
                             note="off-target must NOT switch (specificity control) — SYNTHETIC proxy"))
    pd.DataFrame(rows).to_csv("results/offtarget_controls.csv", index=False)
    print(f"wrote results/offtarget_controls.csv: {len(rows)} off-target control rows (SYNTHETIC)")
    print("On Colab: re-run two-state modeling with each off-target; confirm dynamic_range ~ 1 (no switching).")
else:
    print("Run notebook 04 first to produce results/top_constructs.csv.")

## 4 · (Stretch) Multiplexing concept `[stretch]`

Detecting several analytes at once needs **orthogonal** reporters (different luciferase colors / FRET
pairs), spatial separation (bead/array), or barcoded constructs — plus **cross-talk** controls
(each sensor tested against the others' analytes). Sketch the panel; do not over-claim. This is a
*concept* deliverable, not measured data.

In [ ]:
multiplex_note = """Multiplexing concept (stretch) — sketch, not data:
- Orthogonal readouts: pair each analyte's sensor with a distinguishable reporter (e.g. different
  luciferase substrates/colors, or distinct FRET donor/acceptor pairs).
- Spatial encoding: immobilize sensors on addressable beads/array spots (one analyte per spot).
- Cross-talk controls (MANDATORY): test every sensor against every other analyte; quantify spillover.
- Decide read-out scheme (ratiometric / spectral unmixing) and the blank+off-target controls per channel.
"""
open("results/multiplexing_concept.md", "w").write(multiplex_note)
print("wrote results/multiplexing_concept.md (concept only — no fabricated multiplex data).")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: luminescence/FRET **dose-response**, LOD *estimate from data*, expression, timeline, costed reagents.
- [ ] Controls specified: **no-analyte / blank** (OFF floor + LOD) and **off-target** (`results/offtarget_controls.csv`); positive/calibrator.
- [ ] Dose-response *planning* curve is labeled EXAMPLE_DATA; **no fabricated LOD/luminescence** reported as real.
- [ ] (Stretch) multiplexing concept with cross-talk controls (`results/multiplexing_concept.md`).
- [ ] Honest framing: coupling binding to a clean ON/OFF is hard; report measured dynamic range + LOD, including failures.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — an integrated **binder + switch** biosensor with a functional-readout plan, built on the
binder-family workflow.